## Лабораторная работа 5: Корреляция и свертка

### Упражнение 5.2: Взаимная корреляция и свертка

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import SinSignal, read_wave
from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np

### 1. Взаимная корреляция двух сигналов

In [ ]:
# Создание двух синусоидальных сигналов с разными фазами
signal1 = SinSignal(freq=440, offset=0)
signal2 = SinSignal(freq=440, offset=np.pi/4)

wave1 = signal1.make_wave(duration=0.5, framerate=10000)
wave2 = signal2.make_wave(duration=0.5, framerate=10000)

# Визуализация сигналов
segment1 = wave1.segment(duration=0.01)
segment2 = wave2.segment(duration=0.01)

plt.figure(figsize=(10, 4))
plt.plot(segment1.ts, segment1.ys, label='Сигнал 1')
plt.plot(segment2.ts, segment2.ys, label='Сигнал 2')
plt.legend()
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

In [ ]:
# Вычисление взаимной корреляции
cross_corr = np.correlate(wave1.ys, wave2.ys, mode='same')
cross_corr = cross_corr / cross_corr.max()

# Визуализация
N = len(cross_corr)
lags = np.arange(-N//2, N//2)
plt.plot(lags[:1000], cross_corr[N//2:N//2+1000])
decorate(xlabel='Сдвиг (отсчеты)', ylabel='Взаимная корреляция')
plt.show()

**Вопрос 1:** Как взаимная корреляция показывает фазовый сдвиг между сигналами?

### 2. Свертка сигналов

In [ ]:
# Создание простого импульсного отклика (фильтр)
impulse_response = np.array([0.2, 0.5, 0.2])

# Создание тестового сигнала
signal = SinSignal(freq=440)
wave = signal.make_wave(duration=0.1, framerate=10000)

# Применение свертки
filtered = np.convolve(wave.ys, impulse_response, mode='same')

# Визуализация
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(wave.ts[:200], wave.ys[:200])
plt.title('Исходный сигнал')
decorate(xlabel='Время (с)', ylabel='Амплитуда')

plt.subplot(1, 2, 2)
plt.plot(wave.ts[:200], filtered[:200])
plt.title('После свертки')
decorate(xlabel='Время (с)', ylabel='Амплитуда')

plt.tight_layout()
plt.show()

**Вопрос 2:** Как свертка изменяет сигнал?

### 3. Применение к реальному сигналу

In [ ]:
# Загрузка реального аудиосигнала
wave = read_wave('../lab-1/92002__jcveliz__violin-origional.wav')
segment = wave.segment(start=1.2, duration=0.1)

# Вычисление автокорреляции
corr = np.correlate(segment.ys, segment.ys, mode='same')
corr = corr / corr.max()

# Визуализация
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
segment.plot()
plt.title('Сигнал скрипки')
decorate(xlabel='Время (с)', ylabel='Амплитуда')

plt.subplot(1, 2, 2)
N = len(corr)
lags = np.arange(-N//2, N//2)
plt.plot(lags[:500], corr[N//2:N//2+500])
plt.title('Автокорреляция')
decorate(xlabel='Сдвиг (отсчеты)', ylabel='Корреляция')

plt.tight_layout()
plt.show()

### Упражнение 5.3: Определение высоты тона

In [ ]:
def detect_pitch(wave, min_freq=50, max_freq=2000):
    """
    Определение основной частоты (высоты тона) с помощью автокорреляции
    """
    # Вычисление автокорреляции
    corr = np.correlate(wave.ys, wave.ys, mode='same')
    corr = corr / corr.max()
    
    # Определение диапазона поиска
    min_period = int(wave.framerate / max_freq)
    max_period = int(wave.framerate / min_freq)
    
    # Поиск максимума в нужном диапазоне
    N = len(corr)
    center = N // 2
    
    search_range = corr[center + min_period:center + max_period]
    if len(search_range) > 0:
        peak_idx = np.argmax(search_range)
        period = min_period + peak_idx
        frequency = wave.framerate / period
        return frequency
    return None

# Применение к сигналу скрипки
pitch = detect_pitch(segment)
print(f"Определенная высота тона: {pitch:.2f} Гц")

**Вопрос 3:** Почему автокорреляция эффективна для определения высоты тона?

### Задания для самостоятельной работы

1. Реализуйте функцию для определения задержки между двумя сигналами с помощью взаимной корреляции.
2. Создайте различные импульсные отклики и исследуйте их влияние на сигнал через свертку.
3. Примените определение высоты тона к различным музыкальным инструментам.
4. Исследуйте, как шум влияет на точность определения высоты тона.

In [ ]:
# Место для вашего кода
